# Word Embeddings für den Spamklassifikator: Lösung

Sehen wir uns Word Embeddings an. Dazu wollen wir

1. das [TensorFlow-Tutorial](https://www.tensorflow.org/tutorials/text/word_embeddings) auf den Spam/Ham-Datensatz anwenden.

2. die Embeddings in den [Embedding Projector](http://projector.tensorflow.org/) laden, um das Word Embedding zu visualisieren.

In diesem Notebook findest du eine Lösung dafür, wie du die Spam/Ham-Daten lädst und vorbereitest, um diese NLP-Modelle anzuwenden.

In [ ]:
# alle benötigten Bibliotheken und Funktionen importieren
import io
import re
import string
import tensorflow as tf

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization

from tensorflow import keras
import matplotlib.pyplot as plt
from IPython.display import clear_output

## Daten mit pandas laden

Wir laden unsere Spam- und Ham-Daten mit pandas und teilen unsere Daten anschließend wie gewohnt mit der Funktion train_test_split von sklearn in einen Trainings-, einen Validierungs- und einen Testdatensatz auf.


In [ ]:
# Spam/Ham-Daten laden
dataframe_full = pd.read_csv(
    "../data/SMSSpamCollection.txt",
    encoding="utf-8",
    header=None,
    delimiter="\t",
    names=["target", "text"],
)

# Zielvariable kodieren
dataframe_full["target"] = np.where(dataframe_full["target"] == "spam", 1, 0)

In [ ]:
# Ein erster Blick auf die Daten
dataframe_full.sample(5)

In [ ]:
# Daten in Trainings-, Validierungs- und Testdatensatz aufteilen
dataframe_train, dataframe_test = train_test_split(
    dataframe_full, test_size=0.2, random_state=42
)
dataframe_train, dataframe_val = train_test_split(
    dataframe_train, test_size=0.25, random_state=42
)

In [ ]:
def print_shape(dataframe):
    """Anzahl der Beobachtungen und Anzahl der Spalten des übergebenen DataFrames ausgeben.

    Args:
        dataframe (pandas DataFrame): Ein beliebiger pandas DataFrame
    """
    name = [x for x in globals() if globals()[x] is dataframe][0]
    print(
        f"There are {dataframe.shape[0]} observations and {dataframe.shape[1]} columns in the {name}."
    )

In [ ]:
for i in [dataframe_train, dataframe_val, dataframe_test]:
    print_shape(i)

## pandas-DataFrames in tf.dataset umwandeln

Nach dem Aufteilen der Daten müssen wir sie in „Tensoren“ von TensorFlow umwandeln.

In [ ]:
dataset_train = tf.data.Dataset.from_tensor_slices(
    (dataframe_train.text, dataframe_train.target)
)
dataset_val = tf.data.Dataset.from_tensor_slices(
    (dataframe_val.text, dataframe_val.target)
)
dataset_test = tf.data.Dataset.from_tensor_slices(
    (dataframe_test.text, dataframe_test.target)
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

dataset_train = dataset_train.cache().prefetch(buffer_size=AUTOTUNE)
dataset_val = dataset_val.cache().prefetch(buffer_size=AUTOTUNE)

## Textvorverarbeitung

In [ ]:
def custom_standardization(input_data):
    """Textvorverarbeitung: Kleinschreibung, keine Satzzeichen

    Args:
        input_data (tf.dataframe): [Text, formatiert als tf.string]

    Returns:
        [tf.dataframe]: [vorverarbeiteter Text]
    """
    text_lower = tf.strings.lower(input_data)
    return tf.strings.regex_replace(
        text_lower, "[%s]" % re.escape(string.punctuation), ""
    )

In [ ]:
# Größe des Vokabulars und Anzahl der Wörter in einer Sequenz.
vocab_size = 7546  # aus Notebook 1 übernommen
sequence_length = int(dataframe_train.text.apply(lambda x: len(x.split())).max())

# Die Schicht zur Textvektorisierung nutzen, um Strings zu normalisieren, aufzuteilen und
# auf Ganzzahlen abzubilden. Beachte, dass die Schicht die oben definierte Funktion custom_standardization verwendet.
# Maximale Sequenzlänge festlegen, da nicht alle Beobachtungen gleich lang sind.
vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

In [ ]:
# Einen reinen Textdatensatz (ohne Labels) erstellen, dann adapt aufrufen
train_text = dataset_train.map(lambda x, y: x)
vectorize_layer.adapt(train_text)

In [ ]:
print("1287 ---> ", vectorize_layer.get_vocabulary()[1287])
print(" 313 ---> ", vectorize_layer.get_vocabulary()[313])
print("Vocabulary size: {}".format(len(vectorize_layer.get_vocabulary())))

## Klassifikationsmodell erstellen

Im Tutorial wurde die Batchgröße beim Laden der Daten als tf.Dataset festgelegt. Deshalb müssen wir sie jetzt ebenfalls festlegen. Das ist besonders für das Training des Modells wichtig.
Du kannst die Batches so erstellen, wie hier gezeigt: [Methode tf.data.Dataset.batch(), kombiniert mit der Methode repeat()](https://www.gcptutorials.com/article/how-to-use-batch-method-in-tensorflow).

In [ ]:
dataset_train_batch = dataset_train.repeat().batch(batch_size=32)
dataset_val_batch = dataset_val.repeat().batch(batch_size=32)
dataset_test_batch = dataset_test.repeat().batch(batch_size=32)

In [ ]:
# Form und Typ des in Batches aufgeteilten Datensatzes prüfen
dataset_train_batch

**Modellstruktur festlegen:**
- zuerst die Daten vektorisieren
- dann eine Embedding-Schicht verwenden
- die Schicht globalaveragepooling1D gibt einen Ausgabevektor fester Länge zurück, auch wenn die Eingabe unterschiedlich lang sein kann
- vollständig verbundene Schicht
- letzte Schicht mit einem einzigen Ausgabeknoten

In [ ]:
# Modellstruktur
embedding_dim = 16
model = Sequential(
    [
        vectorize_layer,
        Embedding(vocab_size, embedding_dim, name="embedding"),
        GlobalAveragePooling1D(),
        Dense(16, activation="relu"),
        Dense(1),
    ]
)

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

In [ ]:
# Modell mit dem Adam-Optimizer und BinaryCrossentropy als Verlustfunktion kompilieren
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

In [ ]:
# Verlust des Modells während des Trainings darstellen, erstellt von Daniel: https://medium.com/geekculture/how-to-plot-model-loss-while-training-in-tensorflow-9fa1a1875a5
class PlotLearning(keras.callbacks.Callback):
    """
    Callback, der die Lernkurven des Modells während des Trainings darstellt.
    """

    def on_train_begin(self, logs={}):
        self.metrics = {}
        for metric in logs:
            self.metrics[metric] = []

    def on_epoch_end(self, epoch, logs={}):
        # Metriken speichern
        for metric in logs:
            if metric in self.metrics:
                self.metrics[metric].append(logs.get(metric))
            else:
                self.metrics[metric] = [logs.get(metric)]

        # Darstellung
        metrics = [x for x in logs if "val" not in x]

        f, axs = plt.subplots(1, len(metrics), figsize=(15, 5))
        clear_output(wait=True)

        for i, metric in enumerate(metrics):
            axs[i].plot(range(1, epoch + 2), self.metrics[metric], label=metric)
            if logs["val_" + metric]:
                axs[i].plot(
                    range(1, epoch + 2),
                    self.metrics["val_" + metric],
                    label="val_" + metric,
                )

            axs[i].legend()
            axs[i].grid()

        plt.tight_layout()
        plt.show()

In [ ]:
# Modell trainieren
callbacks_list = [PlotLearning()]
model.fit(
    dataset_train_batch,
    validation_data=dataset_val_batch,
    epochs=20,
    steps_per_epoch=240,
    validation_steps=25,
    callbacks=callbacks_list,
)

In [ ]:
# Verlust und Accuracy auf dem Testdatensatz berechnen.
loss, accuracy = model.evaluate(dataset_test_batch, verbose=2, steps=25)
print(f"Model accuracy: {accuracy}")

In [ ]:
model.summary()

## Trainierte Word Embeddings abrufen und auf der Festplatte speichern

In [ ]:
weights = model.get_layer("embedding").get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

In [ ]:
out_v = io.open("vectors.tsv", "w", encoding="utf-8")
out_m = io.open("metadata.tsv", "w", encoding="utf-8")

for index, word in enumerate(vocab):
    if index == 0:
        continue  # 0 überspringen, das ist Padding.
    vec = weights[index]
    out_v.write("\t".join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")
out_v.close()
out_m.close()

## Embeddings visualisieren
Um die Embeddings zu visualisieren, lade sie in den Embedding Projector hoch.

Öffne den [Embedding Projector](http://projector.tensorflow.org/) (er kann auch in einer lokalen TensorBoard-Instanz laufen).

Klicke auf „Load data“.

Lade die beiden Dateien hoch, die du oben erstellt hast: vecs.tsv und meta.tsv.

